In [1]:
!pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 100.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-c

In [2]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork

from pgmpy.estimators import ExpertKnowledge

# Import required estimators and inference methods
from pgmpy.estimators.HillClimbSearch import HillClimbSearch
from pgmpy.estimators.StructureScore import BIC
from pgmpy.estimators.MLE import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
cols = [
    "age_group",
    "emp_length",
    "fico_bucket_pct",
    "purpose",
    "annual_inc_disc",
    "home_ownership",
    "issue_quarter",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    "open_acc_disc",
    "pub_rec_disc",
    "revol_bal_disc",
    "revol_util_disc",
    "delinq_2yrs_disc",
    "inq_last_6mths_disc",
    "grade",
    "sub_grade",
    "total_acc_disc",
    "loan_amnt_disc",
    "int_rate_disc",
    "installment_disc",
    "default_indicator",
]


# Define the node ordering.
ordering = [
    "age_group",
    "emp_length",
    "fico_bucket_pct",
    "purpose",
    "annual_inc_disc",
    "home_ownership",
    "issue_quarter",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    "open_acc_disc",
    "pub_rec_disc",
    "revol_bal_disc",
    "revol_util_disc",
    "delinq_2yrs_disc",
    "inq_last_6mths_disc",
    "grade",
    "sub_grade",
    "total_acc_disc",
    "loan_amnt_disc",
    "int_rate_disc",
    "installment_disc",
    "default_indicator",
]

# Load your pre-discretized data.
data = pd.read_csv('rebalanced_age_data_discretized.csv')
data = data[cols]
data_no_na_default = data.dropna(subset=["default_indicator"])
data = data_no_na_default

# Split the data into training and testing sets (here, 5% test split).
train_data, test_data = train_test_split(data, test_size=0.025, random_state=42, stratify=data["default_indicator"])
print("Total data size:", len(data))

# All nodes must be present in the DataFrame 'df'
# For example:
# df = pd.read_csv("your_data.csv")

# Create a forbidden (black) list of edges.
# We do this by saying that an edge (u, v) is forbidden if u comes AFTER v in the ordering.
black_list = []
for i in range(len(ordering)):
    for j in range(len(ordering)):
        # If the potential parent (at index i) comes later than the potential child (at index j),
        # then this edge is forbidden.
        if i > j:
            black_list.append((ordering[i], ordering[j]))

required_edges_list = [
    ("age_group", "fico_bucket_pct"),
    ("age_group", "default_indicator"),
    ("emp_length", "default_indicator"),
    ("annual_inc_disc", "default_indicator"),
    ("fico_bucket_pct", "default_indicator"),
    ("purpose", "default_indicator"),
    ("grade", "sub_grade")]

expert_knowledge_ordering = ExpertKnowledge(forbidden_edges = black_list, required_edges=required_edges_list)

# Initialize the HillClimbSearch with your data and a scoring method.
# Here we use BIC (Bayesian Information Criterion) as the scoring metric.
hc = HillClimbSearch(train_data)

# Estimate the Bayesian Model with the ordering constraint applied via the black_list.
# You can also experiment with white_list constraints if you want to force certain edges.
best_model = hc.estimate(expert_knowledge=expert_knowledge_ordering, scoring_method=BIC(train_data))

# Print the learned structure edges.
print("Learned structure edges:", best_model.edges())

# ----------------------------
# Parameter Learning Section
# ----------------------------

# Create a new BayesianNetwork model with the learned structure.
model = DiscreteBayesianNetwork(best_model.edges())

# Fit the network on the training data using Maximum Likelihood Estimation.
model.fit(train_data, estimator=MaximumLikelihoodEstimator)

# Print the learned CPDs for each node.
print("Learned CPDs:")
for cpd in model.get_cpds():
    print(cpd)

# Prepare test data by removing the target column for inference.
test_features = test_data.drop(columns=["default_indicator"])

predictions_list = []
batch_size = 1000 # Adjust based on available memory
for i in range(0, len(test_features), batch_size):
    batch = test_features.iloc[i:i+batch_size]
    predictions_batch = model.predict(batch)
    predictions_list.append(predictions_batch)

# Combine predictions from all batches
predictions = pd.concat(predictions_list)
# Extract predictions and ground truth for evaluation.
y_pred = predictions["default_indicator"]
y_true = test_data["default_indicator"]

Total data size: 183893


  0%|          | 0/1000000 [00:00<?, ?it/s]

Learned structure edges: [('age_group', 'fico_bucket_pct'), ('age_group', 'default_indicator'), ('age_group', 'emp_length'), ('emp_length', 'default_indicator'), ('emp_length', 'issue_quarter'), ('emp_length', 'purpose'), ('emp_length', 'grade'), ('emp_length', 'home_ownership'), ('emp_length', 'fico_bucket_pct'), ('emp_length', 'int_rate_disc'), ('emp_length', 'loan_amnt_disc'), ('emp_length', 'open_acc_disc'), ('emp_length', 'sub_grade'), ('emp_length', 'total_acc_disc'), ('emp_length', 'revol_util_disc'), ('emp_length', 'inq_last_6mths_disc'), ('emp_length', 'revol_bal_disc'), ('emp_length', 'installment_disc'), ('emp_length', 'delinq_2yrs_disc'), ('emp_length', 'pub_rec_disc'), ('fico_bucket_pct', 'default_indicator'), ('fico_bucket_pct', 'grade'), ('purpose', 'default_indicator'), ('purpose', 'home_ownership'), ('annual_inc_disc', 'default_indicator'), ('issue_quarter', 'inflation_t1_disc'), ('issue_quarter', 'housing_price_t1_disc'), ('issue_quarter', 'unemployment_rate_t1_disc')

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/598 [00:00<?, ?it/s]

In [13]:
# Assume 'train_data' is your training DataFrame and your fitted model is 'model'.

# Get the set of all variable names from the data.
data_variables = set(train_data.columns)

# Get the set of variables (nodes) in your model.
model_variables = set(model.nodes())

# Find the variables that are in the data but not in the model.
missing_variables = data_variables - model_variables

print("Variables in the data but not in the model:", missing_variables)

Variables in the data but not in the model: {'annual_inc_disc'}


In [3]:
count_nan_y_true = y_true.isna().sum()
df = pd.DataFrame({
    'y_true': y_true,
    'y_pred': y_pred
})
print("\nDataFrame with predictions:\n", df)
print("\nCount of NaN values in y_pred:", y_pred.isna().sum())
print("\nCount of NaN values in y_true:", count_nan_y_true)

df_no_na = df.dropna()
accuracy = accuracy_score(df_no_na['y_true'], df_no_na['y_pred'])
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(df_no_na['y_true'], df_no_na['y_pred']))
print("Confusion Matrix:\n", confusion_matrix(df_no_na['y_true'], df_no_na['y_pred']))


DataFrame with predictions:
         y_true  y_pred
123        1.0     0.0
144        0.0     0.0
157        0.0     0.0
175        0.0     0.0
180        0.0     0.0
...        ...     ...
183716     0.0     0.0
183784     1.0     0.0
183863     1.0     0.0
183867     0.0     0.0
183888     1.0     0.0

[4598 rows x 2 columns]

Count of NaN values in y_pred: 0

Count of NaN values in y_true: 0

Classification Accuracy: 0.8316659417137886

Classification Report:
               precision    recall  f1-score   support

         0.0       0.83      1.00      0.91      3824
         1.0       0.50      0.00      0.00       774

    accuracy                           0.83      4598
   macro avg       0.67      0.50      0.46      4598
weighted avg       0.78      0.83      0.76      4598

Confusion Matrix:
 [[3823    1]
 [ 773    1]]


In [4]:
import numpy as np
import pandas as pd
from pgmpy.inference import VariableElimination
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score

# -----------------------------------------------------------
# Function to generate predictions given a threshold
# -----------------------------------------------------------
def generate_predictions(inference, test_data, threshold):
    predictions = []
    # Loop over test instances; for each, drop the target to create evidence.
    for idx, row in test_data.drop(columns=["default_indicator"]).iterrows():
        evidence = row.to_dict()
        posterior = inference.query(variables=['default_indicator'], evidence=evidence)
        # Assume that index 1 corresponds to the positive state.
        pred = 1 if posterior.values[1] > threshold else 0
        predictions.append(pred)
    return pd.Series(predictions, index=test_data.index)

# -----------------------------------------------------------
# Initial Setup: Inference Object and Data Preparation
# -----------------------------------------------------------
# Create an inference object.
inference = VariableElimination(model)

# Ensure no missing values in test_data.
test_data = test_data.dropna()

# Ground truth labels.
y_true = test_data["default_indicator"]

# -----------------------------------------------------------
# Search for the Optimal Threshold Maximizing F1 Score for Class 1
# -----------------------------------------------------------
threshold_candidates = np.linspace(0.0, 1.0, 101)  # 101 candidate thresholds.
best_threshold = None
best_f1 = -np.inf  # Start with a very low value.

for threshold in threshold_candidates:
    predictions = generate_predictions(inference, test_data, threshold)

    # Calculate F1 score for class 1 using the binary setting.
    current_f1 = f1_score(y_true, predictions, pos_label=1, zero_division=0)

    # Update if this threshold gives a better F1 score.
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = threshold

print(f"Optimal threshold for maximizing F1 score for class 1: {best_threshold:.2f}")
print(f"Maximum F1 score for class 1 achieved: {best_f1:.2f}")

# -----------------------------------------------------------
# Evaluate with the Selected Threshold
# -----------------------------------------------------------
optimal_predictions = generate_predictions(inference, test_data, best_threshold)
accuracy = accuracy_score(y_true, optimal_predictions)
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_true, optimal_predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_true, optimal_predictions))

Optimal threshold for maximizing F1 score for class 1: 0.13
Maximum F1 score for class 1 achieved: 0.30

Classification Accuracy: 0.39242024466298875

Classification Report:
              precision    recall  f1-score   support

         0.0       0.90      0.31      0.46      3498
         1.0       0.19      0.82      0.30       671

    accuracy                           0.39      4169
   macro avg       0.54      0.56      0.38      4169
weighted avg       0.78      0.39      0.44      4169

Confusion Matrix:
[[1088 2410]
 [ 123  548]]


In [5]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork

from pgmpy.estimators import ExpertKnowledge

# Import required estimators and inference methods
from pgmpy.estimators.HillClimbSearch import HillClimbSearch
from pgmpy.estimators.StructureScore import BIC
from pgmpy.estimators.MLE import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
from pgmpy.estimators import BayesianEstimator

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
cols = [
    "age_group",
    "emp_length",
    "fico_bucket_pct",
    "purpose",
    "annual_inc_disc",
    "home_ownership",
    "issue_quarter",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    "open_acc_disc",
    "pub_rec_disc",
    "revol_bal_disc",
    "revol_util_disc",
    "delinq_2yrs_disc",
    "inq_last_6mths_disc",
    "grade",
    "sub_grade",
    "total_acc_disc",
    "loan_amnt_disc",
    "int_rate_disc",
    "installment_disc",
    "default_indicator",
]


# Define the node ordering.
ordering = [
    "age_group",
    "emp_length",
    "fico_bucket_pct",
    "purpose",
    "annual_inc_disc",
    "home_ownership",
    "issue_quarter",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    "open_acc_disc",
    "pub_rec_disc",
    "revol_bal_disc",
    "revol_util_disc",
    "delinq_2yrs_disc",
    "inq_last_6mths_disc",
    "grade",
    "sub_grade",
    "total_acc_disc",
    "loan_amnt_disc",
    "int_rate_disc",
    "installment_disc",
    "default_indicator",
]

# Load your pre-discretized data.
data = pd.read_csv('rebalanced_age_data_discretized.csv')
data = data[cols]
data_no_na_default = data.dropna(subset=["default_indicator"])
data = data_no_na_default

# Split the data into training and testing sets (here, 5% test split).
train_data, test_data = train_test_split(data, test_size=0.025, random_state=42, stratify=data["default_indicator"])
print("Total data size:", len(data))

# All nodes must be present in the DataFrame 'df'
# For example:
# df = pd.read_csv("your_data.csv")

# Create a forbidden (black) list of edges.
# We do this by saying that an edge (u, v) is forbidden if u comes AFTER v in the ordering.
black_list = []
for i in range(len(ordering)):
    for j in range(len(ordering)):
        # If the potential parent (at index i) comes later than the potential child (at index j),
        # then this edge is forbidden.
        if i > j:
            black_list.append((ordering[i], ordering[j]))

required_edges_list = [
    ("age_group", "fico_bucket_pct"),
    ("age_group", "default_indicator"),
    ("emp_length", "default_indicator"),
    ("annual_inc_disc", "default_indicator"),
    ("fico_bucket_pct", "default_indicator"),
    ("purpose", "default_indicator"),
    ("grade", "sub_grade")]

expert_knowledge_ordering = ExpertKnowledge(forbidden_edges = black_list, required_edges=required_edges_list)

# Initialize the HillClimbSearch with your data and a scoring method.
# Here we use BIC (Bayesian Information Criterion) as the scoring metric.
hc = HillClimbSearch(train_data)

# Estimate the Bayesian Model with the ordering constraint applied via the black_list.
# You can also experiment with white_list constraints if you want to force certain edges.
best_model = hc.estimate(expert_knowledge=expert_knowledge_ordering, scoring_method=BIC(train_data))

# Print the learned structure edges.
print("Learned structure edges:", best_model.edges())

# ----------------------------
# Parameter Learning Section
# ----------------------------

# Create a new BayesianNetwork model with the learned structure.
model = DiscreteBayesianNetwork(best_model.edges())

# Fit the network on the training data using Maximum Likelihood Estimation.
model.fit(train_data, estimator=BayesianEstimator, prior_type='BDeu')

# Print the learned CPDs for each node.
print("Learned CPDs:")
for cpd in model.get_cpds():
    print(cpd)

# Prepare test data by removing the target column for inference.
test_features = test_data.drop(columns=["default_indicator"])

predictions_list = []
batch_size = 1000 # Adjust based on available memory
for i in range(0, len(test_features), batch_size):
    batch = test_features.iloc[i:i+batch_size]
    predictions_batch = model.predict(batch)
    predictions_list.append(predictions_batch)

# Combine predictions from all batches
predictions = pd.concat(predictions_list)
# Extract predictions and ground truth for evaluation.
y_pred = predictions["default_indicator"]
y_true = test_data["default_indicator"]

Total data size: 183893


  0%|          | 0/1000000 [00:00<?, ?it/s]

Learned structure edges: [('age_group', 'fico_bucket_pct'), ('age_group', 'default_indicator'), ('age_group', 'emp_length'), ('emp_length', 'default_indicator'), ('emp_length', 'issue_quarter'), ('emp_length', 'purpose'), ('emp_length', 'grade'), ('emp_length', 'home_ownership'), ('emp_length', 'fico_bucket_pct'), ('emp_length', 'int_rate_disc'), ('emp_length', 'loan_amnt_disc'), ('emp_length', 'open_acc_disc'), ('emp_length', 'sub_grade'), ('emp_length', 'total_acc_disc'), ('emp_length', 'revol_util_disc'), ('emp_length', 'inq_last_6mths_disc'), ('emp_length', 'revol_bal_disc'), ('emp_length', 'installment_disc'), ('emp_length', 'delinq_2yrs_disc'), ('emp_length', 'pub_rec_disc'), ('fico_bucket_pct', 'default_indicator'), ('fico_bucket_pct', 'grade'), ('purpose', 'default_indicator'), ('purpose', 'home_ownership'), ('annual_inc_disc', 'default_indicator'), ('issue_quarter', 'inflation_t1_disc'), ('issue_quarter', 'housing_price_t1_disc'), ('issue_quarter', 'unemployment_rate_t1_disc')

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/598 [00:00<?, ?it/s]

In [6]:
count_nan_y_true = y_true.isna().sum()
df = pd.DataFrame({
    'y_true': y_true,
    'y_pred': y_pred
})
print("\nDataFrame with predictions:\n", df)
print("\nCount of NaN values in y_pred:", y_pred.isna().sum())
print("\nCount of NaN values in y_true:", count_nan_y_true)

df_no_na = df.dropna()
accuracy = accuracy_score(df_no_na['y_true'], df_no_na['y_pred'])
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(df_no_na['y_true'], df_no_na['y_pred']))
print("Confusion Matrix:\n", confusion_matrix(df_no_na['y_true'], df_no_na['y_pred']))


DataFrame with predictions:
         y_true  y_pred
123        1.0     0.0
144        0.0     0.0
157        0.0     0.0
175        0.0     0.0
180        0.0     0.0
...        ...     ...
183716     0.0     0.0
183784     1.0     0.0
183863     1.0     0.0
183867     0.0     0.0
183888     1.0     0.0

[4598 rows x 2 columns]

Count of NaN values in y_pred: 0

Count of NaN values in y_true: 0

Classification Accuracy: 0.8316659417137886

Classification Report:
               precision    recall  f1-score   support

         0.0       0.83      1.00      0.91      3824
         1.0       0.50      0.00      0.00       774

    accuracy                           0.83      4598
   macro avg       0.67      0.50      0.46      4598
weighted avg       0.78      0.83      0.76      4598

Confusion Matrix:
 [[3823    1]
 [ 773    1]]


In [7]:
import numpy as np
import pandas as pd
from pgmpy.inference import VariableElimination
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score

# -----------------------------------------------------------
# Function to generate predictions given a threshold
# -----------------------------------------------------------
def generate_predictions(inference, test_data, threshold):
    predictions = []
    # Loop over test instances; for each, drop the target to create evidence.
    for idx, row in test_data.drop(columns=["default_indicator"]).iterrows():
        evidence = row.to_dict()
        posterior = inference.query(variables=['default_indicator'], evidence=evidence)
        # Assume that index 1 corresponds to the positive state.
        pred = 1 if posterior.values[1] > threshold else 0
        predictions.append(pred)
    return pd.Series(predictions, index=test_data.index)

# -----------------------------------------------------------
# Initial Setup: Inference Object and Data Preparation
# -----------------------------------------------------------
# Create an inference object.
inference = VariableElimination(model)

# Ensure no missing values in test_data.
test_data = test_data.dropna()

# Ground truth labels.
y_true = test_data["default_indicator"]

# -----------------------------------------------------------
# Search for the Optimal Threshold Maximizing F1 Score for Class 1
# -----------------------------------------------------------
threshold_candidates = np.linspace(0.0, 1.0, 101)  # 101 candidate thresholds.
best_threshold = None
best_f1 = -np.inf  # Start with a very low value.

for threshold in threshold_candidates:
    predictions = generate_predictions(inference, test_data, threshold)

    # Calculate F1 score for class 1 using the binary setting.
    current_f1 = f1_score(y_true, predictions, pos_label=1, zero_division=0)

    # Update if this threshold gives a better F1 score.
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = threshold

print(f"Optimal threshold for maximizing F1 score for class 1: {best_threshold:.2f}")
print(f"Maximum F1 score for class 1 achieved: {best_f1:.2f}")

# -----------------------------------------------------------
# Evaluate with the Selected Threshold
# -----------------------------------------------------------
optimal_predictions = generate_predictions(inference, test_data, best_threshold)
accuracy = accuracy_score(y_true, optimal_predictions)
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_true, optimal_predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_true, optimal_predictions))

Optimal threshold for maximizing F1 score for class 1: 0.13
Maximum F1 score for class 1 achieved: 0.30

Classification Accuracy: 0.3914607819621012

Classification Report:
              precision    recall  f1-score   support

         0.0       0.90      0.31      0.46      3498
         1.0       0.19      0.82      0.30       671

    accuracy                           0.39      4169
   macro avg       0.54      0.56      0.38      4169
weighted avg       0.78      0.39      0.44      4169

Confusion Matrix:
[[1084 2414]
 [ 123  548]]


In [15]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork

from pgmpy.estimators import ExpertKnowledge

# Import required estimators and inference methods
from pgmpy.estimators.HillClimbSearch import HillClimbSearch
from pgmpy.estimators.StructureScore import AIC
from pgmpy.estimators.MLE import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
cols = [
    "age_group",
    "emp_length",
    "fico_bucket_pct",
    "purpose",
    "annual_inc_disc",
    "home_ownership",
    "issue_quarter",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    "open_acc_disc",
    "pub_rec_disc",
    "revol_bal_disc",
    "revol_util_disc",
    "delinq_2yrs_disc",
    "inq_last_6mths_disc",
    "grade",
    "sub_grade",
    "total_acc_disc",
    "loan_amnt_disc",
    "int_rate_disc",
    "installment_disc",
    "default_indicator",
]


# Define the node ordering.
ordering = [
    "age_group",
    "emp_length",
    "fico_bucket_pct",
    "purpose",
    "annual_inc_disc",
    "home_ownership",
    "issue_quarter",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    "open_acc_disc",
    "pub_rec_disc",
    "revol_bal_disc",
    "revol_util_disc",
    "delinq_2yrs_disc",
    "inq_last_6mths_disc",
    "grade",
    "sub_grade",
    "total_acc_disc",
    "loan_amnt_disc",
    "int_rate_disc",
    "installment_disc",
    "default_indicator",
]

# Load your pre-discretized data.
data = pd.read_csv('rebalanced_age_data_discretized.csv')
data = data[cols]
data_no_na_default = data.dropna(subset=["default_indicator"])
data = data_no_na_default

# Split the data into training and testing sets (here, 5% test split).
train_data, test_data = train_test_split(data, test_size=0.025, random_state=42, stratify=data["default_indicator"])
print("Total data size:", len(data))

# All nodes must be present in the DataFrame 'df'
# For example:
# df = pd.read_csv("your_data.csv")

# Create a forbidden (black) list of edges.
# We do this by saying that an edge (u, v) is forbidden if u comes AFTER v in the ordering.
black_list = []
for i in range(len(ordering)):
    for j in range(len(ordering)):
        # If the potential parent (at index i) comes later than the potential child (at index j),
        # then this edge is forbidden.
        if i > j:
            black_list.append((ordering[i], ordering[j]))

required_edges_list = [
    ("age_group", "fico_bucket_pct"),
    ("age_group", "default_indicator"),
    ("emp_length", "default_indicator"),
    ("annual_inc_disc", "default_indicator"),
    ("fico_bucket_pct", "default_indicator"),
    ("purpose", "default_indicator"),
    ("grade", "sub_grade")]

expert_knowledge_ordering = ExpertKnowledge(forbidden_edges = black_list, required_edges=required_edges_list)

# Initialize the HillClimbSearch with your data and a scoring method.
# Here we use BIC (Bayesian Information Criterion) as the scoring metric.
hc = HillClimbSearch(train_data)

# Estimate the Bayesian Model with the ordering constraint applied via the black_list.
# You can also experiment with white_list constraints if you want to force certain edges.
best_model = hc.estimate(expert_knowledge=expert_knowledge_ordering, scoring_method=AIC(train_data))

# Print the learned structure edges.
print("Learned structure edges:", best_model.edges())

# ----------------------------
# Parameter Learning Section
# ----------------------------

# Create a new BayesianNetwork model with the learned structure.
model = DiscreteBayesianNetwork(best_model.edges())

# Fit the network on the training data using Maximum Likelihood Estimation.
model.fit(train_data, estimator=MaximumLikelihoodEstimator)

# Print the learned CPDs for each node.
print("Learned CPDs:")
for cpd in model.get_cpds():
    print(cpd)

# Prepare test data by removing the target column for inference.
test_features = test_data.drop(columns=["default_indicator"])

predictions_list = []
batch_size = 1000 # Adjust based on available memory
for i in range(0, len(test_features), batch_size):
    batch = test_features.iloc[i:i+batch_size]
    predictions_batch = model.predict(batch)
    predictions_list.append(predictions_batch)

# Combine predictions from all batches
predictions = pd.concat(predictions_list)
# Extract predictions and ground truth for evaluation.
y_pred = predictions["default_indicator"]
y_true = test_data["default_indicator"]

Total data size: 183893


  0%|          | 0/1000000 [00:00<?, ?it/s]

Learned structure edges: [('age_group', 'fico_bucket_pct'), ('age_group', 'default_indicator'), ('age_group', 'emp_length'), ('emp_length', 'default_indicator'), ('emp_length', 'issue_quarter'), ('emp_length', 'purpose'), ('emp_length', 'grade'), ('emp_length', 'sub_grade'), ('emp_length', 'fico_bucket_pct'), ('emp_length', 'home_ownership'), ('emp_length', 'int_rate_disc'), ('emp_length', 'loan_amnt_disc'), ('emp_length', 'open_acc_disc'), ('emp_length', 'total_acc_disc'), ('emp_length', 'revol_util_disc'), ('emp_length', 'inq_last_6mths_disc'), ('emp_length', 'revol_bal_disc'), ('emp_length', 'installment_disc'), ('emp_length', 'delinq_2yrs_disc'), ('emp_length', 'pub_rec_disc'), ('emp_length', 'gdp_growth_t1_disc'), ('emp_length', 'annual_inc_disc'), ('fico_bucket_pct', 'default_indicator'), ('fico_bucket_pct', 'grade'), ('fico_bucket_pct', 'issue_quarter'), ('fico_bucket_pct', 'purpose'), ('purpose', 'default_indicator'), ('purpose', 'home_ownership'), ('purpose', 'grade'), ('annua

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/598 [00:00<?, ?it/s]

In [16]:
count_nan_y_true = y_true.isna().sum()
df = pd.DataFrame({
    'y_true': y_true,
    'y_pred': y_pred
})
print("\nDataFrame with predictions:\n", df)
print("\nCount of NaN values in y_pred:", y_pred.isna().sum())
print("\nCount of NaN values in y_true:", count_nan_y_true)

df_no_na = df.dropna()
accuracy = accuracy_score(df_no_na['y_true'], df_no_na['y_pred'])
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(df_no_na['y_true'], df_no_na['y_pred']))
print("Confusion Matrix:\n", confusion_matrix(df_no_na['y_true'], df_no_na['y_pred']))


DataFrame with predictions:
         y_true  y_pred
123        1.0     0.0
144        0.0     0.0
157        0.0     0.0
175        0.0     0.0
180        0.0     0.0
...        ...     ...
183716     0.0     0.0
183784     1.0     0.0
183863     1.0     0.0
183867     0.0     0.0
183888     1.0     0.0

[4598 rows x 2 columns]

Count of NaN values in y_pred: 0

Count of NaN values in y_true: 0

Classification Accuracy: 0.8316659417137886

Classification Report:
               precision    recall  f1-score   support

         0.0       0.83      1.00      0.91      3824
         1.0       0.50      0.00      0.00       774

    accuracy                           0.83      4598
   macro avg       0.67      0.50      0.46      4598
weighted avg       0.78      0.83      0.76      4598

Confusion Matrix:
 [[3823    1]
 [ 773    1]]


In [17]:
import numpy as np
import pandas as pd
from pgmpy.inference import VariableElimination
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score

# -----------------------------------------------------------
# Function to generate predictions given a threshold
# -----------------------------------------------------------
def generate_predictions(inference, test_data, threshold):
    predictions = []
    # Loop over test instances; for each, drop the target to create evidence.
    for idx, row in test_data.drop(columns=["default_indicator"]).iterrows():
        evidence = row.to_dict()
        posterior = inference.query(variables=['default_indicator'], evidence=evidence)
        # Assume that index 1 corresponds to the positive state.
        pred = 1 if posterior.values[1] > threshold else 0
        predictions.append(pred)
    return pd.Series(predictions, index=test_data.index)

# -----------------------------------------------------------
# Initial Setup: Inference Object and Data Preparation
# -----------------------------------------------------------
# Create an inference object.
inference = VariableElimination(model)

# Ensure no missing values in test_data.
test_data = test_data.dropna()

# Ground truth labels.
y_true = test_data["default_indicator"]

# -----------------------------------------------------------
# Search for the Optimal Threshold Maximizing F1 Score for Class 1
# -----------------------------------------------------------
threshold_candidates = np.linspace(0.0, 1.0, 101)  # 101 candidate thresholds.
best_threshold = None
best_f1 = -np.inf  # Start with a very low value.

for threshold in threshold_candidates:
    predictions = generate_predictions(inference, test_data, threshold)

    # Calculate F1 score for class 1 using the binary setting.
    current_f1 = f1_score(y_true, predictions, pos_label=1, zero_division=0)

    # Update if this threshold gives a better F1 score.
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = threshold

print(f"Optimal threshold for maximizing F1 score for class 1: {best_threshold:.2f}")
print(f"Maximum F1 score for class 1 achieved: {best_f1:.2f}")

# -----------------------------------------------------------
# Evaluate with the Selected Threshold
# -----------------------------------------------------------
optimal_predictions = generate_predictions(inference, test_data, best_threshold)
accuracy = accuracy_score(y_true, optimal_predictions)
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_true, optimal_predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_true, optimal_predictions))

Optimal threshold for maximizing F1 score for class 1: 0.13
Maximum F1 score for class 1 achieved: 0.30

Classification Accuracy: 0.39242024466298875

Classification Report:
              precision    recall  f1-score   support

         0.0       0.90      0.31      0.46      3498
         1.0       0.19      0.82      0.30       671

    accuracy                           0.39      4169
   macro avg       0.54      0.56      0.38      4169
weighted avg       0.78      0.39      0.44      4169

Confusion Matrix:
[[1088 2410]
 [ 123  548]]


In [8]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork

from pgmpy.estimators import ExpertKnowledge

# Import required estimators and inference methods
from pgmpy.estimators.HillClimbSearch import HillClimbSearch
from pgmpy.estimators.StructureScore import BIC
from pgmpy.estimators.MLE import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
from pgmpy.estimators import BayesianEstimator

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
cols = [
    "age_group",
    "emp_length",
    "fico_bucket_pct",
    "purpose",
    "annual_inc_disc",
    "home_ownership",
    "issue_quarter",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    "open_acc_disc",
    "pub_rec_disc",
    "revol_bal_disc",
    "revol_util_disc",
    "delinq_2yrs_disc",
    "inq_last_6mths_disc",
    "grade",
    "sub_grade",
    "total_acc_disc",
    "loan_amnt_disc",
    "int_rate_disc",
    "installment_disc",
    "default_indicator",
]


# Define the node ordering.
ordering = [
    "age_group",
    "emp_length",
    "fico_bucket_pct",
    "purpose",
    "annual_inc_disc",
    "home_ownership",
    "issue_quarter",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    "open_acc_disc",
    "pub_rec_disc",
    "revol_bal_disc",
    "revol_util_disc",
    "delinq_2yrs_disc",
    "inq_last_6mths_disc",
    "grade",
    "sub_grade",
    "total_acc_disc",
    "loan_amnt_disc",
    "int_rate_disc",
    "installment_disc",
    "default_indicator",
]

# Load your pre-discretized data.
data = pd.read_csv('rebalanced_age_data_discretized.csv')
data = data[cols]
data_no_na_default = data.dropna(subset=["default_indicator"])
data = data_no_na_default

# Split the data into training and testing sets (here, 5% test split).
train_data, test_data = train_test_split(data, test_size=0.025, random_state=42, stratify=data["default_indicator"])
print("Total data size:", len(data))

# All nodes must be present in the DataFrame 'df'
# For example:
# df = pd.read_csv("your_data.csv")

# Create a forbidden (black) list of edges.
# We do this by saying that an edge (u, v) is forbidden if u comes AFTER v in the ordering.
black_list = []

# For any node other than "default_indicator", add (default_indicator -> node)
for node in ordering:
    if node != "default_indicator":
        black_list.append(("default_indicator", node))

# For any node other than "age_group", add (node -> age_group)
for node in ordering:
    if node != "age_group":
        black_list.append((node, "age_group"))


required_edges_list = [
    ("age_group", "fico_bucket_pct"),
    ("age_group", "default_indicator"),
    ("age_group", "emp_length"),
    ("emp_length", "default_indicator"),
    ("annual_inc_disc", "default_indicator"),
    ("fico_bucket_pct", "default_indicator"),
    ("grade", "sub_grade")]

expert_knowledge_ordering = ExpertKnowledge(forbidden_edges = black_list, required_edges=required_edges_list)

# Initialize the HillClimbSearch with your data and a scoring method.
# Here we use BIC (Bayesian Information Criterion) as the scoring metric.
hc = HillClimbSearch(train_data)

# Estimate the Bayesian Model with the ordering constraint applied via the black_list.
# You can also experiment with white_list constraints if you want to force certain edges.
best_model = hc.estimate(expert_knowledge=expert_knowledge_ordering, scoring_method=BIC(train_data))

# Print the learned structure edges.
print("Learned structure edges:", best_model.edges())

# ----------------------------
# Parameter Learning Section
# ----------------------------

# Create a new BayesianNetwork model with the learned structure.
model = DiscreteBayesianNetwork(best_model.edges())

# Fit the network on the training data using Maximum Likelihood Estimation.
model.fit(train_data, estimator=BayesianEstimator, prior_type='BDeu')

# Print the learned CPDs for each node.
print("Learned CPDs:")
for cpd in model.get_cpds():
    print(cpd)

# Prepare test data by removing the target column for inference.
test_features = test_data.drop(columns=["default_indicator"])

predictions_list = []
batch_size = 1000 # Adjust based on available memory
for i in range(0, len(test_features), batch_size):
    batch = test_features.iloc[i:i+batch_size]
    predictions_batch = model.predict(batch)
    predictions_list.append(predictions_batch)

# Combine predictions from all batches
predictions = pd.concat(predictions_list)
# Extract predictions and ground truth for evaluation.
y_pred = predictions["default_indicator"]
y_true = test_data["default_indicator"]

Total data size: 183893


  0%|          | 0/1000000 [00:00<?, ?it/s]

Learned structure edges: [('age_group', 'fico_bucket_pct'), ('age_group', 'emp_length'), ('age_group', 'default_indicator'), ('emp_length', 'default_indicator'), ('emp_length', 'issue_quarter'), ('emp_length', 'purpose'), ('emp_length', 'grade'), ('emp_length', 'home_ownership'), ('emp_length', 'int_rate_disc'), ('emp_length', 'loan_amnt_disc'), ('emp_length', 'open_acc_disc'), ('emp_length', 'sub_grade'), ('emp_length', 'fico_bucket_pct'), ('emp_length', 'total_acc_disc'), ('emp_length', 'revol_util_disc'), ('emp_length', 'inq_last_6mths_disc'), ('emp_length', 'revol_bal_disc'), ('emp_length', 'installment_disc'), ('emp_length', 'delinq_2yrs_disc'), ('emp_length', 'pub_rec_disc'), ('fico_bucket_pct', 'default_indicator'), ('fico_bucket_pct', 'grade'), ('purpose', 'home_ownership'), ('annual_inc_disc', 'default_indicator'), ('issue_quarter', 'housing_price_t1_disc'), ('issue_quarter', 'unemployment_rate_t1_disc'), ('issue_quarter', 'gdp_growth_t1_disc'), ('issue_quarter', 'fedfunds_t1_

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/598 [00:00<?, ?it/s]

In [9]:
count_nan_y_true = y_true.isna().sum()
df = pd.DataFrame({
    'y_true': y_true,
    'y_pred': y_pred
})
print("\nDataFrame with predictions:\n", df)
print("\nCount of NaN values in y_pred:", y_pred.isna().sum())
print("\nCount of NaN values in y_true:", count_nan_y_true)

df_no_na = df.dropna()
accuracy = accuracy_score(df_no_na['y_true'], df_no_na['y_pred'])
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(df_no_na['y_true'], df_no_na['y_pred']))
print("Confusion Matrix:\n", confusion_matrix(df_no_na['y_true'], df_no_na['y_pred']))


DataFrame with predictions:
         y_true  y_pred
123        1.0     0.0
144        0.0     0.0
157        0.0     0.0
175        0.0     0.0
180        0.0     0.0
...        ...     ...
183716     0.0     0.0
183784     1.0     0.0
183863     1.0     0.0
183867     0.0     0.0
183888     1.0     0.0

[4598 rows x 2 columns]

Count of NaN values in y_pred: 0

Count of NaN values in y_true: 0

Classification Accuracy: 0.8316659417137886

Classification Report:
               precision    recall  f1-score   support

         0.0       0.83      1.00      0.91      3824
         1.0       0.00      0.00      0.00       774

    accuracy                           0.83      4598
   macro avg       0.42      0.50      0.45      4598
weighted avg       0.69      0.83      0.76      4598

Confusion Matrix:
 [[3824    0]
 [ 774    0]]


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [10]:
import numpy as np
import pandas as pd
from pgmpy.inference import VariableElimination
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score

# -----------------------------------------------------------
# Function to generate predictions given a threshold
# -----------------------------------------------------------
def generate_predictions(inference, test_data, threshold):
    predictions = []
    # Loop over test instances; for each, drop the target to create evidence.
    for idx, row in test_data.drop(columns=["default_indicator"]).iterrows():
        evidence = row.to_dict()
        posterior = inference.query(variables=['default_indicator'], evidence=evidence)
        # Assume that index 1 corresponds to the positive state.
        pred = 1 if posterior.values[1] > threshold else 0
        predictions.append(pred)
    return pd.Series(predictions, index=test_data.index)

# -----------------------------------------------------------
# Initial Setup: Inference Object and Data Preparation
# -----------------------------------------------------------
# Create an inference object.
inference = VariableElimination(model)

# Ensure no missing values in test_data.
test_data = test_data.dropna()

# Ground truth labels.
y_true = test_data["default_indicator"]

# -----------------------------------------------------------
# Search for the Optimal Threshold Maximizing F1 Score for Class 1
# -----------------------------------------------------------
threshold_candidates = np.linspace(0.0, 1.0, 101)  # 101 candidate thresholds.
best_threshold = None
best_f1 = -np.inf  # Start with a very low value.

for threshold in threshold_candidates:
    predictions = generate_predictions(inference, test_data, threshold)

    # Calculate F1 score for class 1 using the binary setting.
    current_f1 = f1_score(y_true, predictions, pos_label=1, zero_division=0)

    # Update if this threshold gives a better F1 score.
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = threshold

print(f"Optimal threshold for maximizing F1 score for class 1: {best_threshold:.2f}")
print(f"Maximum F1 score for class 1 achieved: {best_f1:.2f}")

# -----------------------------------------------------------
# Evaluate with the Selected Threshold
# -----------------------------------------------------------
optimal_predictions = generate_predictions(inference, test_data, best_threshold)
accuracy = accuracy_score(y_true, optimal_predictions)
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_true, optimal_predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_true, optimal_predictions))

Optimal threshold for maximizing F1 score for class 1: 0.15
Maximum F1 score for class 1 achieved: 0.30

Classification Accuracy: 0.43847445430558885

Classification Report:
              precision    recall  f1-score   support

         0.0       0.88      0.38      0.53      3498
         1.0       0.19      0.74      0.30       671

    accuracy                           0.44      4169
   macro avg       0.54      0.56      0.42      4169
weighted avg       0.77      0.44      0.49      4169

Confusion Matrix:
[[1330 2168]
 [ 173  498]]


In [11]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork

from pgmpy.estimators import ExpertKnowledge

# Import required estimators and inference methods
from pgmpy.estimators.HillClimbSearch import HillClimbSearch
from pgmpy.estimators.StructureScore import AIC
from pgmpy.estimators.MLE import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
from pgmpy.estimators import BayesianEstimator

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
cols = [
    "age_group",
    "emp_length",
    "fico_bucket_pct",
    "purpose",
    "annual_inc_disc",
    "home_ownership",
    "issue_quarter",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    "open_acc_disc",
    "pub_rec_disc",
    "revol_bal_disc",
    "revol_util_disc",
    "delinq_2yrs_disc",
    "inq_last_6mths_disc",
    "grade",
    "sub_grade",
    "total_acc_disc",
    "loan_amnt_disc",
    "int_rate_disc",
    "installment_disc",
    "default_indicator",
]


# Define the node ordering.
ordering = [
    "age_group",
    "emp_length",
    "fico_bucket_pct",
    "purpose",
    "annual_inc_disc",
    "home_ownership",
    "issue_quarter",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    "open_acc_disc",
    "pub_rec_disc",
    "revol_bal_disc",
    "revol_util_disc",
    "delinq_2yrs_disc",
    "inq_last_6mths_disc",
    "grade",
    "sub_grade",
    "total_acc_disc",
    "loan_amnt_disc",
    "int_rate_disc",
    "installment_disc",
    "default_indicator",
]

# Load your pre-discretized data.
data = pd.read_csv('rebalanced_age_data_discretized.csv')
data = data[cols]
data_no_na_default = data.dropna(subset=["default_indicator"])
data = data_no_na_default

# Split the data into training and testing sets (here, 5% test split).
train_data, test_data = train_test_split(data, test_size=0.025, random_state=42, stratify=data["default_indicator"])
print("Total data size:", len(data))

# All nodes must be present in the DataFrame 'df'
# For example:
# df = pd.read_csv("your_data.csv")

# Create a forbidden (black) list of edges.
# We do this by saying that an edge (u, v) is forbidden if u comes AFTER v in the ordering.
black_list = []

# For any node other than "default_indicator", add (default_indicator -> node)
for node in ordering:
    if node != "default_indicator":
        black_list.append(("default_indicator", node))

# For any node other than "age_group", add (node -> age_group)
for node in ordering:
    if node != "age_group":
        black_list.append((node, "age_group"))


required_edges_list = [
    ("age_group", "fico_bucket_pct"),
    ("age_group", "default_indicator"),
    ("age_group", "emp_length"),
    ("emp_length", "default_indicator"),
    ("annual_inc_disc", "default_indicator"),
    ("fico_bucket_pct", "default_indicator"),
    ("grade", "sub_grade")]

expert_knowledge_ordering = ExpertKnowledge(forbidden_edges = black_list, required_edges=required_edges_list)

# Initialize the HillClimbSearch with your data and a scoring method.
# Here we use BIC (Bayesian Information Criterion) as the scoring metric.
hc = HillClimbSearch(train_data)

# Estimate the Bayesian Model with the ordering constraint applied via the black_list.
# You can also experiment with white_list constraints if you want to force certain edges.
best_model = hc.estimate(expert_knowledge=expert_knowledge_ordering, scoring_method=AIC(train_data))

# Print the learned structure edges.
print("Learned structure edges:", best_model.edges())

# ----------------------------
# Parameter Learning Section
# ----------------------------

# Create a new BayesianNetwork model with the learned structure.
model = DiscreteBayesianNetwork(best_model.edges())

# Fit the network on the training data using Maximum Likelihood Estimation.
model.fit(train_data, estimator=BayesianEstimator, prior_type='BDeu')

# Print the learned CPDs for each node.
print("Learned CPDs:")
for cpd in model.get_cpds():
    print(cpd)

# Prepare test data by removing the target column for inference.
test_features = test_data.drop(columns=["default_indicator"])

predictions_list = []
batch_size = 1000 # Adjust based on available memory
for i in range(0, len(test_features), batch_size):
    batch = test_features.iloc[i:i+batch_size]
    predictions_batch = model.predict(batch)
    predictions_list.append(predictions_batch)

# Combine predictions from all batches
predictions = pd.concat(predictions_list)
# Extract predictions and ground truth for evaluation.
y_pred = predictions["default_indicator"]
y_true = test_data["default_indicator"]

Total data size: 183893


  0%|          | 0/1000000 [00:00<?, ?it/s]

Learned structure edges: [('age_group', 'fico_bucket_pct'), ('age_group', 'emp_length'), ('age_group', 'default_indicator'), ('age_group', 'inflation_t1_disc'), ('emp_length', 'default_indicator'), ('emp_length', 'issue_quarter'), ('emp_length', 'purpose'), ('emp_length', 'grade'), ('emp_length', 'sub_grade'), ('emp_length', 'home_ownership'), ('emp_length', 'inflation_t1_disc'), ('emp_length', 'int_rate_disc'), ('emp_length', 'fico_bucket_pct'), ('emp_length', 'loan_amnt_disc'), ('emp_length', 'open_acc_disc'), ('emp_length', 'total_acc_disc'), ('emp_length', 'revol_util_disc'), ('emp_length', 'inq_last_6mths_disc'), ('emp_length', 'revol_bal_disc'), ('emp_length', 'installment_disc'), ('emp_length', 'delinq_2yrs_disc'), ('emp_length', 'pub_rec_disc'), ('fico_bucket_pct', 'default_indicator'), ('purpose', 'home_ownership'), ('annual_inc_disc', 'default_indicator'), ('annual_inc_disc', 'emp_length'), ('issue_quarter', 'housing_price_t1_disc'), ('issue_quarter', 'unemployment_rate_t1_di

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/598 [00:00<?, ?it/s]

In [12]:
count_nan_y_true = y_true.isna().sum()
df = pd.DataFrame({
    'y_true': y_true,
    'y_pred': y_pred
})
print("\nDataFrame with predictions:\n", df)
print("\nCount of NaN values in y_pred:", y_pred.isna().sum())
print("\nCount of NaN values in y_true:", count_nan_y_true)

df_no_na = df.dropna()
accuracy = accuracy_score(df_no_na['y_true'], df_no_na['y_pred'])
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(df_no_na['y_true'], df_no_na['y_pred']))
print("Confusion Matrix:\n", confusion_matrix(df_no_na['y_true'], df_no_na['y_pred']))


DataFrame with predictions:
         y_true  y_pred
123        1.0     0.0
144        0.0     0.0
157        0.0     0.0
175        0.0     0.0
180        0.0     0.0
...        ...     ...
183716     0.0     1.0
183784     1.0     0.0
183863     1.0     0.0
183867     0.0     0.0
183888     1.0     0.0

[4598 rows x 2 columns]

Count of NaN values in y_pred: 0

Count of NaN values in y_true: 0

Classification Accuracy: 0.8314484558503697

Classification Report:
               precision    recall  f1-score   support

         0.0       0.83      1.00      0.91      3824
         1.0       0.00      0.00      0.00       774

    accuracy                           0.83      4598
   macro avg       0.42      0.50      0.45      4598
weighted avg       0.69      0.83      0.76      4598

Confusion Matrix:
 [[3823    1]
 [ 774    0]]


In [14]:
import numpy as np
import pandas as pd
from pgmpy.inference import VariableElimination
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score

# -----------------------------------------------------------
# Function to generate predictions given a threshold
# -----------------------------------------------------------
def generate_predictions(inference, test_data, threshold):
    predictions = []
    # Loop over test instances; for each, drop the target to create evidence.
    for idx, row in test_data.drop(columns=["default_indicator"]).iterrows():
        evidence = row.to_dict()
        posterior = inference.query(variables=['default_indicator'], evidence=evidence)
        # Assume that index 1 corresponds to the positive state.
        pred = 1 if posterior.values[1] > threshold else 0
        predictions.append(pred)
    return pd.Series(predictions, index=test_data.index)

# -----------------------------------------------------------
# Initial Setup: Inference Object and Data Preparation
# -----------------------------------------------------------
# Create an inference object.
inference = VariableElimination(model)

# Ensure no missing values in test_data.
test_data = test_data.dropna()

# Ground truth labels.
y_true = test_data["default_indicator"]

# -----------------------------------------------------------
# Search for the Optimal Threshold Maximizing F1 Score for Class 1
# -----------------------------------------------------------
threshold_candidates = np.linspace(0.0, 1.0, 101)  # 101 candidate thresholds.
best_threshold = None
best_f1 = -np.inf  # Start with a very low value.

for threshold in threshold_candidates:
    predictions = generate_predictions(inference, test_data, threshold)

    # Calculate F1 score for class 1 using the binary setting.
    current_f1 = f1_score(y_true, predictions, pos_label=1, zero_division=0)

    # Update if this threshold gives a better F1 score.
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = threshold

print(f"Optimal threshold for maximizing F1 score for class 1: {best_threshold:.2f}")
print(f"Maximum F1 score for class 1 achieved: {best_f1:.2f}")

# -----------------------------------------------------------
# Evaluate with the Selected Threshold
# -----------------------------------------------------------
optimal_predictions = generate_predictions(inference, test_data, best_threshold)
accuracy = accuracy_score(y_true, optimal_predictions)
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_true, optimal_predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_true, optimal_predictions))

Optimal threshold for maximizing F1 score for class 1: 0.15
Maximum F1 score for class 1 achieved: 0.30

Classification Accuracy: 0.43847445430558885

Classification Report:
              precision    recall  f1-score   support

         0.0       0.88      0.38      0.53      3498
         1.0       0.19      0.74      0.30       671

    accuracy                           0.44      4169
   macro avg       0.54      0.56      0.42      4169
weighted avg       0.77      0.44      0.49      4169

Confusion Matrix:
[[1330 2168]
 [ 173  498]]
